In [18]:
import json
import sys
import dotenv

dotenv.load_dotenv()
sys.path.insert(0, "..")

In [19]:
with open("data/tiny_dataset.json") as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} questions")

from baseline_bot import get_binary_gpt_prediction

Loaded 24 questions


In [ ]:
question = dataset[-1]['question']
resolution_date = dataset[-1]['resolution_date']

print(question)
print(resolution_date)
print('{resolution_date}' in question)

print(question)

According to Wikipedia, will a vaccine have been developed for Mycoplasma pneumonia by {resolution_date}?
2026-02-08
True
According to Wikipedia, will a vaccine have been developed for Mycoplasma pneumonia by 2026-02-08?


In [30]:
# Run baseline bot sequentially across questions.
# num_runs=1 for speed; increase for more stable estimates.
NUM_RUNS = 4

prediction_date = "2025-09-09T00:00:00Z"

predictions = []
for i, entry in enumerate(dataset):

    question_details = {
        "title": entry["question"],
        "resolution_criteria": entry["resolution_criteria"],
        "description": entry["background"],
        "fine_print": entry.get("source_intro", ""),
    }

    question = entry["question"]
    question = question.replace("{resolution_date}", resolution_date)
    question = question.replace("{forecast_due_date}", prediction_date)
    question_details["title"] = question

    print(f"[{i+1}/{len(dataset)}] {question_details['title']}...")
    try:
        probability, comment = await get_binary_gpt_prediction(question_details, 
                                                               num_runs=NUM_RUNS, 
                                                               prediction_date=prediction_date)
        try:
            market_price = float(entry["freeze_datetime_value"])
        except (ValueError, TypeError):
            market_price = None
        
        predictions.append({
            "id": entry["id"],
            "question": question_details["title"],
            "prediction": probability,
            "market_price": market_price,
            "resolved_to": entry["resolved_to"],
            "comment": comment,
        })
        print(f"  → {probability:.1%}")

    except Exception as e:
        print(f"  ERROR: {e}")
        predictions.append({
            "id": entry["id"],
            "question": question_details["title"],
            "prediction": None,
            "market_price": None,
            "resolved_to": entry["resolved_to"],
            "comment": str(e),
        })

[1/24] Will there be at least one podium sweep at the 2026 Winter Olympic Games?...
  → 35.0%
[2/24] Will Jayden Daniels win the 2025–26 NFL OPOY?...
  → 30.0%
[3/24] Will there be more 'Riots' in Finland for the 30 days before 2026-02-08 compared to the 30-day average of 'Riots' over the 360 days preceding 2025-09-09T00:00:00Z?

e.g. If the forecast due date is 2024-01-01 and we have the following data:
Date,'Riots'
2023-11-11,1
2023-10-10,2
to calculate the 30-day average of 'Riots' over the preceding 360 days, we’d have: (1+2)/12=0.25.

In this example, for the question to resolve positively, 1 or more 'Riots' would need to occur in the 30 days leading up to the resolution....
  → 35.0%
[4/24] Will there be more than ten times as many 'Protests' in Russia for the 30 days before 2026-02-08 compared to one plus the 30-day average of 'Protests' over the 360 days preceding 2025-09-09T00:00:00Z?

e.g. If the forecast due date is 2024-01-01 and we have the following data:
Date,'Protests'


In [31]:
question

'According to Wikipedia, will a vaccine have been developed for Mycoplasma pneumonia by 2026-02-08?'

In [32]:
with open("baseline_predictions_ensemble.json", "w") as f:
    json.dump(predictions, f, indent=2)

successes = sum(1 for p in predictions if p["prediction"] is not None)
print(f"Saved {successes}/{len(predictions)} successful predictions to baseline_predictions_ensemble.json")

Saved 24/24 successful predictions to baseline_predictions_ensemble.json
